# MT03 — The Gymnasium API & reward signal

### Lab Description

Reinforcement-learning algorithms expect a standard interface: `reset()` returns an observation, and `step(action)` returns `(observation, reward, terminated, truncated, info)`. robosuite ships a **`GymWrapper`** that exposes any task through this **Gymnasium** API with a flat observation vector and a `Box` action space — ready to plug into a learning algorithm.

Equally important is the **reward**: with `reward_shaping=True` the Lift task gives a *dense* signal that grows as the arm approaches and lifts the cube (instead of a single sparse +1 at success). This lab shows both the API and what the reward signal looks like during a reach.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium) that this notebook is built for.

## Goals
- Wrap a robosuite task with the Gymnasium API via `GymWrapper`
- Read the observation/action spaces and the 5-tuple `step` return
- Understand dense (shaped) vs sparse reward
- Plot the reward signal over a scripted reach

### Imports and renderer

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import mujoco
import imageio
import robosuite as suite
from robosuite import load_composite_controller_config
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)  # hide repetitive controller-config warnings

os.makedirs("output/videos", exist_ok=True)
print("robosuite", suite.__version__, "| mujoco", mujoco.__version__)

# robosuite's built-in camera-observation renderer can emit intermittently
# corrupted frames on this ROCm/EGL stack. We instead drive mujoco.Renderer
# directly (the reliable path) and show only the visual geom group (group 1),
# which avoids the green/blue collision shapes and looks correct.
def make_renderer(env, height=256, width=256):
    R = mujoco.Renderer(env.sim.model._model, height=height, width=width)
    opt = mujoco.MjvOption()
    opt.geomgroup[:] = 0
    opt.geomgroup[1] = 1
    return R, opt

def grab_frame(env, R, opt, camera="frontview"):
    mujoco.mj_forward(env.sim.model._model, env.sim.data._data)
    R.update_scene(env.sim.data._data, camera=camera, scene_option=opt)
    return R.render()

### The Gymnasium interface

`GymWrapper` turns the dict observation into a single flat vector and exposes `observation_space` / `action_space`. `reset()` returns `(obs, info)` and `step(a)` returns `(obs, reward, terminated, truncated, info)` — the contract every Gymnasium-compatible RL library relies on.

In [ ]:
from robosuite.wrappers import GymWrapper

controller = load_composite_controller_config(controller="BASIC", robot="Panda")
gym_env = GymWrapper(suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                     has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                     reward_shaping=True, control_freq=20, horizon=200))
obs, info = gym_env.reset()
print("observation_space:", gym_env.observation_space.shape)
print("action_space     :", gym_env.action_space.shape)

obs, reward, terminated, truncated, info = gym_env.step(gym_env.action_space.sample())
print("step -> obs", obs.shape, "| reward", round(float(reward), 3),
      "| terminated", terminated, "| truncated", truncated)
gym_env.close()

### What the reward signal looks like

With reward shaping on, the reward is **dense**: it rises smoothly as the gripper nears the cube and jumps when the cube is lifted. We drive a simple scripted reach (end-effector toward the cube) and record the reward each step — the rising curve is exactly the signal a learning agent would climb. We also record frames to watch the reach.

In [ ]:
import matplotlib.pyplot as plt

controller = load_composite_controller_config(controller="BASIC", robot="Panda")
env = suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                 has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                 reward_shaping=True, control_freq=20, horizon=200)
obs = env.reset()
R, opt = make_renderer(env, 256, 256)
rewards, frames = [], []
for _ in range(120):
    delta = np.clip((obs["cube_pos"] - obs["robot0_eef_pos"]) * 5, -1, 1)
    obs, reward, done, info = env.step(np.array([delta[0], delta[1], delta[2], 0, 0, 0, -1.0]))
    rewards.append(reward)
    frames.append(grab_frame(env, R, opt, camera="frontview"))
R.close()

plt.figure(figsize=(7, 3))
plt.plot(rewards); plt.xlabel("step"); plt.ylabel("shaped reward")
plt.title("Dense reward rises as the arm reaches the cube"); plt.grid(True); plt.show()
imageio.mimsave("output/videos/mt03_reach.mp4", frames, fps=20)
print("mean shaped reward over reach:", round(float(np.mean(rewards)), 3))

### Watch the reach

In [ ]:
Video(url="output/videos/mt03_reach.mp4")

## Conclusions

You exposed a robosuite task through the standard Gymnasium API and saw the dense reward signal a learner would optimize. Hand-scripting got the arm to the cube, but writing controllers by hand does not scale. In MT04 we collect demonstrations from this scripted expert and train a PyTorch neural network to reproduce the behaviour — **behavior cloning**.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT